# Retrieve SWOT LR L3 Unsmoothed data from AVISO by cycle/pass

This notebook adapts the AVISO/CNES cookbook to search AVISO THREDDS for SWOT LR L3 Unsmoothed NetCDF files, select by **cycle number** and **pass number**, use latitude/longitude bounds defaulting to the **whole globe**, save retrieved subsets as local `.nc` files, reopen them with `xarray`, and visualize the result.

## Environment
Required packages: `xarray`, `numpy`, `siphon`, `pydap`, `requests`, `matplotlib`, and optionally `cartopy`.

In [ ]:
# Uncomment if needed in a conda/mamba environment:
# !mamba install -q -c conda-forge xarray numpy siphon pydap requests matplotlib cartopy netcdf4

## Imports

In [ ]:
import os
import re
import warnings
import stat
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import requests as rq
import xarray as xr
import shutil
from siphon.catalog import TDSCatalog
from xarray.backends import PydapDataStore

import matplotlib.pyplot as plt


try:
    import cartopy.crs as ccrs
    CARTOPY_AVAILABLE = True
except Exception:
    CARTOPY_AVAILABLE = False

warnings.filterwarnings("ignore")

# Aviso Login

In [ ]:
# Load .env from current directory
load_dotenv()

username = os.getenv("AVISO_USERNAME")
password = os.getenv("AVISO_PASSWORD")

if not username or not password:
    raise ValueError(
        "AVISO_USERNAME and AVISO_PASSWORD must be set in .env"
    )

netrc = Path.home() / ".netrc"

netrc.write_text(
    f"machine tds-odatis.aviso.altimetry.fr "
    f"login {username} "
    f"password {password}\n"
)

# Required by requests/netrc on many systems
os.chmod(netrc, stat.S_IRUSR | stat.S_IWUSR)

print(f"Created {netrc}")

In [ ]:
session = rq.Session()
session.trust_env = True

r = session.get(
    "https://tds-odatis.aviso.altimetry.fr/thredds/catalog.html"
)

print(r.status_code)

## User parameters
Set `CYCLE_NUMBER` and `PASS_NUMBER` for the single cycle/pass you want. For this initial notebook, the latitude/longitude bounds cover the whole globe.

If you want the pass number to equal the cycle number, leave `PASS_NUMBER = CYCLE_NUMBER`; otherwise set it manually.

In [ ]:
URL_CATALOGUE = (
    "https://tds-odatis.aviso.altimetry.fr/thredds/catalog/"
    "dataset-l3-swot-karin-nadir-validated/l3_lr_ssh/v2_0_1/Expert/catalog.html"
)

In [ ]:
def download_swot_clip_to_netcdf(
    cycle_number: int,
    pass_number: int,
    lat_range: tuple[float, float],
    lon_range: tuple[float, float],
    output_file: str | Path,
    level: str = "L3",
    subset: str = "Expert",
    url_catalogue: str = (
        "https://tds-odatis.aviso.altimetry.fr/thredds/catalog/"
        "dataset-l3-swot-karin-nadir-validated/l3_lr_ssh/v2_0_1/Expert/catalog.html"
    ),
) -> list[Path]:
    """
    Find one SWOT L3 LR SSH file for one cycle/pass, clip by lat/lon box,
    and save the clipped section to output_file.
    """

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    subsets = ["Basic", "Expert", "Unsmoothed", "WindWave", "Technical"]
    regex = re.compile(rf"SWOT_{level}_LR_SSH_{subset}_(\d+)_(\d+)_.*\.nc$")

    def crawl_catalog_pruned(catalog):
        for ds in catalog.datasets.values():
            yield ds

        for ref_name, ref in catalog.catalog_refs.items():
            cycle_match = re.search(r"cycle_(\d+)", ref_name)
            if cycle_match:
                cycle = int(cycle_match.group(1))
                if cycle != cycle_number:
                    continue

            if ref_name in subsets and ref_name != subset:
                continue

            try:
                subcat = ref.follow()
            except Exception:
                continue

            yield from crawl_catalog_pruned(subcat)

    def dataset_matches(dataset) -> bool:
        if not dataset.access_urls or "OPENDAP" not in dataset.access_urls:
            return False
        if not dataset.url_path:
            return False

        filename = dataset.url_path.rsplit("/", 1)[-1]
        match = regex.match(filename)
        if match is None:
            return False

        file_cycle = int(match.group(1))
        file_pass = int(match.group(2))

        return file_cycle == cycle_number and file_pass == pass_number

    def open_remote_dataset(dataset_url: str) -> xr.Dataset:
        session = rq.Session()
        store = PydapDataStore.open(
            dataset_url,
            session=session,
            timeout=300,
            user_charset="UTF-8",
        )
        return xr.open_dataset(store)

    def normalize_longitudes(lon: xr.DataArray):
        lo, hi = lon_range

        if float(lon.max()) > 180 and lo < 0:
            return lon % 360, (lo % 360, hi % 360)

        return lon, lon_range

    def make_lon_mask(lon: xr.DataArray) -> xr.DataArray:
        lon_norm, normalized_range = normalize_longitudes(lon)
        lo, hi = normalized_range

        if lo <= hi:
            return (lon_norm >= lo) & (lon_norm <= hi)

        return (lon_norm >= lo) | (lon_norm <= hi)

    def get_line_indexes(ds_coords: xr.Dataset) -> tuple[int, int]:
        lat_ok = (
            (ds_coords["latitude"] >= lat_range[0])
            & (ds_coords["latitude"] <= lat_range[1])
        )
        lon_ok = make_lon_mask(ds_coords["longitude"])

        mask = (lat_ok & lon_ok).any("num_pixels")
        idx = np.where(mask.values)[0]

        if idx.size == 0:
            raise ValueError("No SWOT lines intersect the requested lat/lon range.")

        return int(idx[0]), int(idx[-1]) + 1

    catalog = TDSCatalog(url_catalogue)
    matching_datasets = [
        ds for ds in crawl_catalog_pruned(catalog)
        if dataset_matches(ds)
    ]

    print(f"Number of matching datasets: {len(matching_datasets)}")

    if len(matching_datasets) != 1:
        raise ValueError(
            f"Expected exactly 1 matching dataset for cycle={cycle_number}, "
            f"pass={pass_number}, but found {len(matching_datasets)}"
        )

    if output_file.exists():
        print(f"Using cached file: {output_file}")
        return [output_file]

    dataset_node = matching_datasets[0]
    dataset_url = dataset_node.access_urls["OPENDAP"]

    print(f"Reading coordinates: {dataset_url}")

    ds_coords = open_remote_dataset(dataset_url + "?latitude,longitude")
    idx_first, idx_last = get_line_indexes(ds_coords)
    ds_coords.close()

    print(f"Reading data lines [{idx_first}:{idx_last}] from {dataset_node.name}")

    remote_ds = open_remote_dataset(dataset_url)

    subset_ds = remote_ds.isel(
        num_lines=slice(idx_first, idx_last)
    )

    subset_ds.attrs["source_opendap_url"] = dataset_url
    subset_ds.attrs["cycle_number"] = cycle_number
    subset_ds.attrs["pass_number"] = pass_number
    subset_ds.attrs["lat_range"] = str(lat_range)
    subset_ds.attrs["lon_range"] = str(lon_range)

    subset_ds.load()
    subset_ds.to_netcdf(output_file)

    remote_ds.close()
    subset_ds.close()

    print(f"Wrote: {output_file}")

    return [output_file]

In [ ]:
file_path = (
    "SWOT_IW_Labeled_Dataset/Eastern Equatorial Indian/data/"
    "SWOT_L2_LR_SSH_Expert_005_105_20231016T094434_20231016T103603_PGC0_01.nc"
)

ds = xr.open_dataset(file_path)

lat_min = float(ds.lat.min())
lat_max = float(ds.lat.max())

lon_min = float(ds.lon.min())
lon_max = float(ds.lon.max())

print(f"Latitude range: ({lat_min:.3f}, {lat_max:.3f})")
print(f"Longitude range: ({lon_min:.3f}, {lon_max:.3f})")

lat_range = (lat_min, lat_max)
lon_range = (lon_min, lon_max)

In [ ]:
SRC_ROOT = Path("SWOT_IW_Labeled_Dataset")
DST_ROOT = Path("SWOT_L3_IW_Labeled_Dataset")


def get_lat_lon_range(nc_file: Path):
    ds = xr.open_dataset(nc_file)

    lat_name = "latitude" if "latitude" in ds else "lat"
    lon_name = "longitude" if "longitude" in ds else "lon"

    lat = ds[lat_name].values
    lon = ds[lon_name].values

    lat_range = [float(lat.min()), float(lat.max())]
    lon_range = [float(lon.min()), float(lon.max())]

    ds.close()
    return lat_range, lon_range


def parse_cycle_pass(nc_file: Path):
    # Example:
    # SWOT_L2_LR_SSH_Expert_001_478_20230807T063402_...
    match = re.search(r"Expert_(\d{3})_(\d{3})_", nc_file.name)
    if not match:
        raise ValueError(f"Could not parse cycle/pass from {nc_file.name}")

    cycle = int(match.group(1))
    swot_pass = int(match.group(2))
    return cycle, swot_pass

def fix_l3_copy_against_original(
    original_file: Path,
    copied_file: Path,
    original_var: str = "ssha",
    copied_var: str = "ssha_unfiltered",
):
    original_file = Path(original_file)
    copied_file = Path(copied_file)

    with xr.open_dataset(original_file) as ds_orig:
        orig_data = ds_orig[original_var].values
        target_shape = orig_data.shape

    with xr.open_dataset(copied_file) as ds_copy:
        ds_copy = ds_copy.load()

    if copied_var not in ds_copy:
        raise ValueError(f"{copied_var} not found in {copied_file}")

    copy_data = ds_copy[copied_var].values

    target_lines, target_pixels = target_shape
    copy_lines, copy_pixels = copy_data.shape

    if copy_pixels != target_pixels:
        raise ValueError(
            f"num_pixels mismatch: original={target_pixels}, copied={copy_pixels}"
        )

    # Fix num_lines by trimming or padding
    if copy_lines > target_lines:
        print(f"Trimming num_lines: {copy_lines} -> {target_lines}")
        ds_copy = ds_copy.isel(num_lines=slice(0, target_lines))
        copy_data = ds_copy[copied_var].values

    elif copy_lines < target_lines:
        print(f"Padding num_lines: {copy_lines} -> {target_lines}")

        pad_count = target_lines - copy_lines
        pad_block = np.full(
            (pad_count, target_pixels),
            np.nan,
            dtype=copy_data.dtype,
        )

        padded_data = np.concatenate([copy_data, pad_block], axis=0)

        ds_copy = ds_copy.reindex(
            num_lines=np.arange(target_lines),
            fill_value=np.nan,
        )

        ds_copy[copied_var].values[:] = padded_data
        copy_data = ds_copy[copied_var].values

    # Fill missing L3 pixels with original L2 SSHA values
    orig_valid = ~np.isnan(orig_data)
    copy_missing = np.isnan(copy_data)

    fill_mask = orig_valid & copy_missing
    fill_count = int(fill_mask.sum())

    if fill_count > 0:
        print(
            f"Filling {fill_count:,} missing pixels in {copied_var} "
            f"using {original_var}"
        )
        copy_data[fill_mask] = orig_data[fill_mask]

    ds_copy[copied_var].values[:] = copy_data

    # Final check
    final_missing = int((~np.isnan(orig_data) & np.isnan(ds_copy[copied_var].values)).sum())

    if final_missing > 0:
        raise ValueError(
            f"Still missing {final_missing:,} pixels after filling."
        )

    tmp_file = copied_file.with_suffix(".tmp.nc")
    ds_copy.to_netcdf(tmp_file)

    ds_copy.close()

    copied_file.unlink()
    tmp_file.rename(copied_file)

    print(f"Fixed file written: {copied_file}")

for region_dir in SRC_ROOT.iterdir():
    if not region_dir.is_dir():
        continue

    src_data_dir = region_dir / "data"
    src_label_dir = region_dir / "bb_labels"

    dst_region_dir = DST_ROOT / region_dir.name
    dst_data_dir = dst_region_dir / "data"
    dst_label_dir = dst_region_dir / "bb_labels"

    dst_data_dir.mkdir(parents=True, exist_ok=True)

    # Copy labels
    if src_label_dir.exists():
        if dst_label_dir.exists():
            shutil.rmtree(dst_label_dir)
        shutil.copytree(src_label_dir, dst_label_dir)

    # Process NetCDF files
    for nc_file in src_data_dir.glob("*.nc"):
        lat_range, lon_range = get_lat_lon_range(nc_file)
        cycle_number, pass_number = parse_cycle_pass(nc_file)

        output_path = dst_data_dir / nc_file.name

        print(f"Processing {region_dir.name}")
        print(f"  file: {nc_file.name}")
        print(f"  cycle: {cycle_number}, pass: {pass_number}")
        print(f"  lat_range: {lat_range}")
        print(f"  lon_range: {lon_range}")
        print(f"  output: {output_path}")

        download_swot_clip_to_netcdf(
            cycle_number=cycle_number,
            pass_number=pass_number,
            lat_range=lat_range,
            lon_range=lon_range,
            output_file=str(output_path),
        )

        fix_l3_copy_against_original(
            original_file=nc_file,
            copied_file=output_path,
            original_var="ssha",
            copied_var="ssha_unfiltered",
        )

In [ ]:
file_path = "SWOT_IW_Labeled_Dataset/Andaman Sea/data/SWOT_L2_LR_SSH_Expert_005_202_20231019T205456_20231019T214624_PGC0_01.nc"

ds = xr.open_dataset(file_path)

var = "ssha"
limit = np.nanpercentile(np.abs(ds[var]), 99)

plt.figure(figsize=(12, 6))
plt.scatter(
    ds["lon"].values.ravel(),
    ds["lat"].values.ravel(),
    c=ds[var].values.ravel(),
    s=1,
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
)
plt.colorbar(label=var)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"{var} from SWOT L3")
plt.grid(True)
plt.show()

In [ ]:
nc_path = "SWOT_L3_IW_Labeled_Dataset/Andaman Sea/data/SWOT_L2_LR_SSH_Expert_005_202_20231019T205456_20231019T214624_PGC0_01.nc"

ds = xr.open_dataset(nc_path)

var = "ssha_unfiltered"

plt.figure(figsize=(12, 6))
plt.scatter(
    ds["longitude"].values.ravel(),
    ds["latitude"].values.ravel(),
    c=ds[var].values.ravel(),
    s=1,
    cmap="RdBu_r",
)
plt.colorbar(label=var)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"{var} from SWOT L3")
plt.grid(True)
plt.show()

In [ ]:
ds